# Test YOLO Deteksi & Counting — Google Colab (GPU)

Notebook ini menguji model `best.pt` di Google Colab dengan GPU.

**Sebelum mulai, upload ke Google Drive:**
```
MyDrive/
└── pktj/
    ├── models/
    │   └── best.pt
    └── data/
        ├── images/
        │   ├── train/   ← 142 gambar
        │   └── val/     ← 25 gambar
        └── labels/
            ├── train/
            └── val/
```

**Pastikan Runtime → Change runtime type → GPU (T4)**

## Cell 1 — Cek GPU & Install Dependensi

In [ ]:
# Cek GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU tersedia:')
    print(result.stdout.split('\n')[8])  # baris info GPU
else:
    print('GPU tidak tersedia — ganti runtime ke GPU!')

# Install dependensi
!pip install ultralytics --quiet
!pip install pyyaml --quiet

import torch
print(f'\nPyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice yang akan digunakan: {DEVICE}')

## Cell 2 — Mount Google Drive & Setup Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ======================================================
# SESUAIKAN jika folder di Drive kamu berbeda
DRIVE_ROOT  = Path('/content/drive/MyDrive/pktj')
# ======================================================

MODEL_PATH   = DRIVE_ROOT / 'models' / 'best.pt'
DATASET_DIR  = DRIVE_ROOT / 'data'
VAL_IMAGES   = DATASET_DIR / 'images' / 'val'
VAL_LABELS   = DATASET_DIR / 'labels' / 'val'
TRAIN_IMAGES = DATASET_DIR / 'images' / 'train'
TRAIN_LABELS = DATASET_DIR / 'labels' / 'train'
OUTPUT_DIR   = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)

CLASS_NAMES  = {0: 'mobil', 1: 'bus', 2: 'truk'}
CLASS_COLORS = {
    'mobil': (0, 255, 0),
    'bus':   (255, 100, 0),
    'truk':  (0, 100, 255),
}

print('=== Cek File ===')
print(f'Model      : {MODEL_PATH} — {"OK" if MODEL_PATH.exists() else "TIDAK ADA!"}')
print(f'Val images : {VAL_IMAGES} — {"OK" if VAL_IMAGES.exists() else "TIDAK ADA!"}')
print(f'Train imgs : {TRAIN_IMAGES} — {"OK" if TRAIN_IMAGES.exists() else "TIDAK ADA!"}')

val_imgs   = list(VAL_IMAGES.glob('*.jpg'))   + list(VAL_IMAGES.glob('*.png'))   if VAL_IMAGES.exists() else []
train_imgs = list(TRAIN_IMAGES.glob('*.jpg')) + list(TRAIN_IMAGES.glob('*.png')) if TRAIN_IMAGES.exists() else []
print(f'\nJumlah val images  : {len(val_imgs)}')
print(f'Jumlah train images: {len(train_imgs)}')

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'best.pt tidak ditemukan di {MODEL_PATH}. Upload ke Drive dulu!')

## Cell 3 — Load Model

In [ ]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
print(f'Model loaded : {MODEL_PATH.name}')
print(f'Device       : {DEVICE}')
print(f'Classes      : {model.names}')

# Warmup
import numpy as np
dummy = np.zeros((640, 640, 3), dtype=np.uint8)
model.predict(dummy, device=DEVICE, verbose=False)
print('Warmup selesai — model siap')

## Cell 4 — Analisis Distribusi Label Dataset

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

def hitung_label(label_dir):
    counter = Counter()
    for lf in Path(label_dir).glob('*.txt'):
        for line in lf.read_text().strip().splitlines():
            if line:
                cls_id = int(line.split()[0])
                counter[CLASS_NAMES.get(cls_id, str(cls_id))] += 1
    return counter

train_count = hitung_label(TRAIN_LABELS)
val_count   = hitung_label(VAL_LABELS)

print('=== Ground Truth Dataset ===')
print(f'Train: {dict(train_count)}  → total {sum(train_count.values())} objek')
print(f'Val  : {dict(val_count)}    → total {sum(val_count.values())} objek')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (title, cnt) in zip(axes, [('Train (142 img)', train_count), ('Val (25 img)', val_count)]):
    keys = list(CLASS_NAMES.values())
    vals = [cnt.get(k, 0) for k in keys]
    bars = ax.bar(keys, vals, color=['#2ecc71', '#3498db', '#e67e22'])
    ax.set_title(f'Distribusi Kelas — {title}', fontweight='bold')
    ax.set_ylabel('Jumlah Objek')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', fontweight='bold')
plt.tight_layout()
out = OUTPUT_DIR / 'distribusi.png'
plt.savefig(str(out), dpi=120)
plt.show()
print(f'Saved: {out}')

## Cell 5 — Deteksi Seluruh Val Set vs Ground Truth

In [ ]:
import time

CONF = 0.25
pred_counter = Counter()
gt_counter   = Counter()
per_image    = []

t0 = time.time()
for i, img_path in enumerate(sorted(val_imgs)):
    lbl_path = VAL_LABELS / (img_path.stem + '.txt')
    gt = Counter()
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            if line:
                gt[CLASS_NAMES.get(int(line.split()[0]), '?')] += 1

    results = model.predict(str(img_path), device=DEVICE, conf=CONF, verbose=False)
    pred = Counter()
    if results[0].boxes is not None:
        for cls_id in results[0].boxes.cls.cpu().numpy().astype(int):
            pred[CLASS_NAMES.get(cls_id, str(cls_id))] += 1

    pred_counter += pred
    gt_counter   += gt
    per_image.append({'file': img_path.name, 'gt': dict(gt), 'pred': dict(pred)})

elapsed = time.time() - t0
fps = len(val_imgs) / elapsed

print(f'=== Hasil Deteksi Val Set ({len(val_imgs)} gambar) ===')
print(f'Waktu   : {elapsed:.1f}s  |  {fps:.1f} img/s  ({DEVICE.upper()})')
print()
print(f'{"Kelas":<8} {"GT":>6} {"Pred":>6} {"Selisih":>8}')
print('-' * 32)
for cls in CLASS_NAMES.values():
    gt_n   = gt_counter.get(cls, 0)
    pred_n = pred_counter.get(cls, 0)
    diff   = pred_n - gt_n
    sign   = '+' if diff >= 0 else ''
    print(f'{cls:<8} {gt_n:>6} {pred_n:>6} {sign+str(diff):>8}')
print('-' * 32)
print(f'{"Total":<8} {sum(gt_counter.values()):>6} {sum(pred_counter.values()):>6}')

## Cell 6 — Visualisasi Bounding Box (6 Sampel)

In [ ]:
import cv2

SAMPLE = 6
samples = sorted(val_imgs)[:SAMPLE]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, img_path in enumerate(samples):
    frame = cv2.imread(str(img_path))
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w  = frame.shape[:2]

    # Ground truth (abu-abu)
    lbl_path = VAL_LABELS / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            if not line:
                continue
            parts = line.split()
            cls_id = int(parts[0])
            cx, cy = float(parts[1])*w, float(parts[2])*h
            bw, bh = float(parts[3])*w, float(parts[4])*h
            x1, y1 = int(cx - bw/2), int(cy - bh/2)
            x2, y2 = int(cx + bw/2), int(cy + bh/2)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (200, 200, 200), 1)
            cv2.putText(frame, f'GT:{CLASS_NAMES.get(cls_id,"?")}',
                        (x1, max(y1-4, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)

    # Prediksi (warna)
    results  = model.predict(str(img_path), device=DEVICE, conf=CONF, verbose=False)
    n_det    = 0
    if results[0].boxes is not None:
        for box, cls_id, conf in zip(
            results[0].boxes.xyxy.cpu().numpy(),
            results[0].boxes.cls.cpu().numpy().astype(int),
            results[0].boxes.conf.cpu().numpy()
        ):
            x1, y1, x2, y2 = map(int, box)
            cls_name = CLASS_NAMES.get(cls_id, str(cls_id))
            color    = CLASS_COLORS.get(cls_name, (255, 255, 0))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f'{cls_name} {conf:.2f}',
                        (x1, max(y1-8, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            n_det += 1

    axes[i].imshow(frame)
    axes[i].set_title(f'{img_path.name}  |  deteksi: {n_det}', fontsize=9)
    axes[i].axis('off')

fig.suptitle('Prediksi (warna) vs Ground Truth (abu-abu)', fontsize=13, fontweight='bold')
plt.tight_layout()
out = OUTPUT_DIR / 'visualisasi.png'
plt.savefig(str(out), dpi=120)
plt.show()
print(f'Saved: {out}')

## Cell 7 — Simulasi Counting (Crossing Line)

Frame-frame train dipakai sebagai simulasi urutan frame video.
Kendaraan dihitung saat centroid melewati garis tengah frame.

In [ ]:
train_list = sorted(train_imgs)

# Tracking state
track_history = {}
next_id       = 0
count_total   = 0
count_kiri    = Counter()
count_kanan   = Counter()

MAX_DIST   = 100
MIN_FRAMES = 2

print(f'Simulasi counting pada {len(train_list)} frame (GPU: {DEVICE.upper()})...')
t0 = time.time()

# Batch predict untuk kecepatan GPU
BATCH = 8
batches = [train_list[i:i+BATCH] for i in range(0, len(train_list), BATCH)]

frame_idx = 0
for batch_paths in batches:
    batch_results = model.predict(
        [str(p) for p in batch_paths],
        device=DEVICE, conf=CONF, verbose=False
    )
    for img_path, res in zip(batch_paths, batch_results):
        frame = cv2.imread(str(img_path))
        if frame is None:
            continue
        h, w = frame.shape[:2]
        LINE_Y = h // 2

        detections = []
        if res.boxes is not None:
            for box, cls_id, conf in zip(
                res.boxes.xyxy.cpu().numpy(),
                res.boxes.cls.cpu().numpy().astype(int),
                res.boxes.conf.cpu().numpy()
            ):
                x1, y1, x2, y2 = box
                cx, cy = int((x1+x2)/2), int((y1+y2)/2)
                detections.append({
                    'cx': cx, 'cy': cy,
                    'cls': CLASS_NAMES.get(cls_id, 'mobil'),
                    'conf': float(conf)
                })

        matched = set()
        for det in detections:
            best_id, best_dist = None, MAX_DIST
            for tid, hist in track_history.items():
                if tid in matched:
                    continue
                lx, ly = hist['positions'][-1]
                d = np.sqrt((det['cx']-lx)**2 + (det['cy']-ly)**2)
                if d < best_dist:
                    best_dist, best_id = d, tid

            if best_id is not None:
                track_history[best_id]['positions'].append((det['cx'], det['cy']))
                track_history[best_id]['cls'] = det['cls']
                matched.add(best_id)
                tid = best_id
            else:
                tid = next_id; next_id += 1
                track_history[tid] = {
                    'positions': [(det['cx'], det['cy'])],
                    'cls': det['cls'], 'counted': False
                }
                matched.add(tid)

            hist = track_history[tid]
            if not hist['counted'] and len(hist['positions']) >= MIN_FRAMES:
                prev_cy = hist['positions'][-2][1]
                curr_cy = hist['positions'][-1][1]
                if prev_cy <= LINE_Y and curr_cy > LINE_Y:
                    hist['counted'] = True
                    count_total += 1
                    lane = 'kiri' if det['cx'] < w//2 else 'kanan'
                    (count_kiri if lane == 'kiri' else count_kanan)[det['cls']] += 1

        frame_idx += 1

elapsed = time.time() - t0
fps = len(train_list) / elapsed

print(f'Selesai: {elapsed:.1f}s  |  {fps:.1f} img/s')
print()
print('=== Hasil Counting ===')
print(f'Total kendaraan : {count_total}')
print()
print(f'{"Kelas":<8} {"Kiri":>6} {"Kanan":>6} {"Total":>6}')
print('-' * 30)
for cls in CLASS_NAMES.values():
    k  = count_kiri.get(cls, 0)
    ka = count_kanan.get(cls, 0)
    print(f'{cls:<8} {k:>6} {ka:>6} {k+ka:>6}')
print('-' * 30)
print(f'{"Total":<8} {sum(count_kiri.values()):>6} {sum(count_kanan.values()):>6} {count_total:>6}')

## Cell 8 — Visualisasi Hasil Counting

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
classes = list(CLASS_NAMES.values())
colors  = ['#2ecc71', '#3498db', '#e67e22']
totals  = [count_kiri.get(c,0) + count_kanan.get(c,0) for c in classes]

# Bar total per kelas
bars = axes[0].bar(classes, totals, color=colors)
axes[0].set_title('Total per Kelas', fontweight='bold')
axes[0].set_ylabel('Jumlah')
for bar, v in zip(bars, totals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(v), ha='center', fontweight='bold')

# Grouped bar per lajur
x     = np.arange(len(classes))
width = 0.35
axes[1].bar(x - width/2, [count_kiri.get(c,0) for c in classes],  width, label='Kiri',  color='#1abc9c')
axes[1].bar(x + width/2, [count_kanan.get(c,0) for c in classes], width, label='Kanan', color='#9b59b6')
axes[1].set_title('Per Lajur', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(classes)
axes[1].legend()
axes[1].set_ylabel('Jumlah')

# Pie
if sum(totals) > 0:
    axes[2].pie(totals, labels=classes, colors=colors, autopct='%1.1f%%', startangle=90)
axes[2].set_title('Proporsi Kelas', fontweight='bold')

fig.suptitle(f'Hasil Counting — Total: {count_total} kendaraan', fontsize=13, fontweight='bold')
plt.tight_layout()
out = OUTPUT_DIR / 'counting.png'
plt.savefig(str(out), dpi=120)
plt.show()
print(f'Saved: {out}')

## Cell 9 — Evaluasi Model (mAP, Precision, Recall)

In [ ]:
import yaml

data_yaml_content = {
    'path': str(DATASET_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'names': {0: 'mobil', 1: 'bus', 2: 'truk'}
}
tmp_yaml = Path('/content/data_eval.yaml')
with open(tmp_yaml, 'w') as f:
    yaml.dump(data_yaml_content, f)

print('Menjalankan evaluasi... (lebih cepat dengan GPU)\n')
t0 = time.time()

metrics = model.val(
    data=str(tmp_yaml),
    device=DEVICE,
    conf=0.25,
    iou=0.5,
    verbose=True,
    plots=False,
    save=False,
)

elapsed = time.time() - t0
print(f'\nWaktu evaluasi: {elapsed:.1f}s')
print()
print('=== Metrik Evaluasi ===')
try:
    mp      = metrics.box.mp
    mr      = metrics.box.mr
    map50   = metrics.box.map50
    map5095 = metrics.box.map
    print(f'Precision (mean) : {mp:.4f}  ({mp*100:.1f}%)')
    print(f'Recall (mean)    : {mr:.4f}  ({mr*100:.1f}%)')
    print(f'mAP@0.5          : {map50:.4f}  ({map50*100:.1f}%)')
    print(f'mAP@0.5:0.95     : {map5095:.4f}  ({map5095*100:.1f}%)')
    print()
    if map50 >= 0.8:
        verdict = 'BAGUS — siap produksi'
    elif map50 >= 0.6:
        verdict = 'CUKUP — bisa improve dengan data lebih banyak'
    elif map50 >= 0.4:
        verdict = 'KURANG — perlu lebih banyak data & augmentasi'
    else:
        verdict = 'BURUK — perlu training ulang dari awal'
    print(f'Verdict: {verdict}')
except Exception as e:
    print(f'Tidak bisa baca metrik: {e}')

## Cell 10 — Download Semua Output ke Komputer

In [ ]:
from google.colab import files

output_files = list(OUTPUT_DIR.glob('*.png'))
print(f'File output yang tersedia ({len(output_files)}):')
for f in output_files:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}  ({size_kb:.0f} KB)')

print()
download = input('Download semua ke komputer? (y/n): ').strip().lower()
if download == 'y':
    for f in output_files:
        files.download(str(f))
    print('Selesai download.')
else:
    print('Skip download.')

# ============================
print()
print('=== RINGKASAN AKHIR ===')
print(f'Device          : {DEVICE.upper()}')
print(f'Val images      : {len(val_imgs)}')
print(f'Train images    : {len(train_imgs)}')
print()
print('Ground Truth Val:')
for c in CLASS_NAMES.values():
    print(f'  {c:<8}: {val_count.get(c, 0)}')
print()
print('Prediksi Val:')
for c in CLASS_NAMES.values():
    print(f'  {c:<8}: {pred_counter.get(c, 0)}')
print()
print('Counting (train frames):')
print(f'  Total    : {count_total}')
for c in CLASS_NAMES.values():
    k  = count_kiri.get(c, 0)
    ka = count_kanan.get(c, 0)
    print(f'  {c:<8}: {k+ka} (kiri:{k}, kanan:{ka})')
try:
    print()
    print(f'mAP@0.5   : {map50*100:.1f}%')
    print(f'Precision : {mp*100:.1f}%')
    print(f'Recall    : {mr*100:.1f}%')
except:
    pass